# Fine-tune Laya PT/ES as the System 1 answer selector

Task: given a visitor question and 3-5 candidate topic labels from e5-base, pick the topic that answers it, or `ninguna de estas opciones`.

1. Runtime → GPU (L4 or T4).
2. Run the cells in order. Cell 3 asks for `laya_ft.zip` (from `data/knowledge/laya_ft/`).
3. Cell 4 is a 1-minute GPU smoke (200 rows). If it fails, stop and paste the error.
4. Cell 5 trains, then zips the checkpoint, copies it to Drive and downloads it **before** any evaluation.
5. Cell 6 evaluates on the GPU and downloads the predictions.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install laya==0.5.1

In [ ]:
# Run A = Spanish/Portuguese NLI checkpoint (default). Optional run B = multilingual base:
# set BASE = "B" and rerun from cell 4.
BASE = "A"
EPOCHS = 4

In [ ]:
from google.colab import files
import zipfile, os, json, time, shutil
up = files.upload()  # choose laya_ft.zip
zipfile.ZipFile(next(iter(up)), "r").extractall("/content/laya_ft")
print(os.listdir("/content/laya_ft"))
print(open("/content/laya_ft/stats.json").read())

In [ ]:
import laya, torch
from laya.train import TrainConfig, dry_run, finetune, resolve_checkpoint_dir
from huggingface_hub import snapshot_download

if BASE == "A":
    base_dir = resolve_checkpoint_dir("telepatia-ai/laya-pt-es-nli")
else:
    base_dir = os.path.join(snapshot_download("convaiinnovations/laya", allow_patterns=["multilingual/*"]), "multilingual")

# GPU smoke: 200 rows, 1 epoch, then load and predict. Catches an end-of-run crash in ~1 minute.
lines = open("/content/laya_ft/train.jsonl").read().splitlines()
open("/content/smoke.jsonl", "w").write("\n".join(lines[:200]) + "\n")
smoke_cfg = TrainConfig(epochs=1, micro_batch=16, grad_accum=1, shuffle_options=("choice",), calib_max=40)
finetune("/content/smoke.jsonl", base_dir, "/content/smoke_ckpt", smoke_cfg, device="cuda")
r = json.loads(lines[0])
print("SMOKE OK:", laya.load("/content/smoke_ckpt", device="cuda", vegaml=False).system_one(r["state"], r["questions"])["answers"]["topic"]["choice"])
shutil.rmtree("/content/smoke_ckpt")

# ~2,150 optimizer updates (8.6k items / 16 x 4 epochs); the library warns on prior collapse when too few.
cfg = TrainConfig(epochs=EPOCHS, micro_batch=16, grad_accum=1, shuffle_options=("choice",),
                  log_every=100, eval_data="/content/laya_ft/dev.jsonl")
print(json.dumps(dry_run("/content/laya_ft/train.jsonl", base_dir, cfg), default=str)[:1500])

In [ ]:
OUT = f"/content/laya_ft_ckpt_{BASE}"
t0 = time.time()
summary = finetune("/content/laya_ft/train.jsonl", base_dir, OUT, cfg, device="cuda")
print("train minutes:", round((time.time() - t0) / 60, 1))
print(json.dumps({k: summary.get(k) for k in ("train_items", "skipped", "temperature", "epoch_loss", "eval_items")}, default=str)[:2000])
# Persist first. Zip only what laya.load needs (skip the duplicate checkpoint_latest/, ~644 MB more).
zip_path = f"/content/laya_ft_ckpt_{BASE}.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_STORED) as z:
    for root, _, fs in os.walk(OUT):
        if "checkpoint_latest" in root:
            continue
        for f in fs:
            full = os.path.join(root, f)
            z.write(full, os.path.relpath(full, OUT))
print("zip MB:", round(os.path.getsize(zip_path) / 1e6, 1))
try:
    from google.colab import drive
    drive.mount("/content/drive")
    shutil.copy(zip_path, "/content/drive/MyDrive/")
    print("copied to Drive: MyDrive/" + os.path.basename(zip_path))
except Exception as exc:  # Drive is a backup; the download below is the main path
    print("Drive copy skipped:", exc)
files.download(zip_path)

In [ ]:
import numpy as np
NONE = "ninguna de estas opciones"
agent = laya.load(OUT, device="cuda", vegaml=False)
rows = [json.loads(l) for l in open("/content/laya_ft/eval.jsonl")]
preds = []
for r in rows:
    a = agent.system_one(r["state"], r["questions"])["answers"]["topic"]
    preds.append({"kind": r["meta"]["kind"], "expected": r["expected"]["topic"], "heldout_gold": r["meta"]["heldout_gold"],
                  "first_option": r["questions"]["topic"]["criteria"][0], "choice": a["choice"], "probs": a["probabilities"]})
para = [p for p in preds if p["kind"] == "paraphrase"]
ood = [p for p in preds if p["kind"] != "paraphrase"]
acc = lambda xs, key="choice": round(float(np.mean([p[key] == p["expected"] for p in xs])), 4) if xs else None
print("paraphrase: Laya", acc(para), "| always option 1 (= e5-base top-1)", acc(para, "first_option"))
print("held-out entries: Laya", acc([p for p in para if p["heldout_gold"]]), "| option 1", acc([p for p in para if p["heldout_gold"]], "first_option"))
print("out-of-domain says none:", acc(ood))
with open(f"/content/laya_ft_eval_preds_{BASE}.jsonl", "w") as f:
    f.write("".join(json.dumps(p, ensure_ascii=False) + "\n" for p in preds))
files.download(f"/content/laya_ft_eval_preds_{BASE}.jsonl")